# New high-altitude dataset: independent review

Development split only. Review saved results and raw drone images; no fitting, threshold search or locked-test evaluation. Existing result configurations remain separate. Notebook outputs are the analysis record.

In [1]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
while not (root / "single_photo_pipeline.py").exists():
    if root == root.parent:
        raise RuntimeError("Repository not found")
    root = root.parent
folder = root / "tests/Review_Highalt_20261118_codex"
output = folder / "results"
sys.path.insert(0, str(folder))
from review_blocks import *
source = root / "tests/Test_Iteration_12/results"
images = root / "dataset/Dataset_Official_3/d1/images"
development, metadata, split = load_development(images / "metadata.csv", source / "highalt_split.csv")
print("All metadata:", len(metadata), "Development:", len(development), "Locked images excluded:", int((split.split != "dev").sum()))
print("Height range:", metadata.alt.min(), metadata.alt.max(), "median", metadata.alt.median())
display(development.groupby("band", observed=True).alt.agg(["size", "min", "max"]))

All metadata: 500 Development: 332 Locked images excluded: 168
Height range: 100.599998 2016.599976 median 709.449982


,size,min,max
band,,,
100-300,74,100.599998,298.799988
1000-1300,43,1002.400024,1299.300049
1300-1600,45,1300.800049,1536.000000
1600-2100,42,1607.000000,2016.599976
300-500,48,301.399994,462.600006
500-800,68,515.299988,791.700012
800-1000,12,816.200012,985.700012


In [2]:
stage_path = next((source / "stageeval/tables_highalt_dev332").glob("per_photo*.csv"))
stages = pd.read_csv(stage_path).sort_values("photo").reset_index(drop=True)
assert np.allclose(stages.height, development.alt, atol=1e-5)
assert len(stages) == len(development)
summary = summarise_stages(stages)
summary.to_csv(output / "stage_by_height_codex.csv", index=False)
display(summary.round(3))
print("Single-query K100/M25 score", 6 * stages.points.mean(), "wrong", int((stages.points < 0).sum()))
headings = pd.read_csv(next((source / "stageeval/tables_highalt_dev332").glob("headings_per_photo*.csv")))
comparison = []
for variant, group in headings.groupby("variant", sort=False):
    comparison.append(dict(variant=variant, photos=len(group), score60=6*group.points.mean(), wrong=int((group.points < 0).sum()), answered=int(group.answered.sum())))
display(pd.DataFrame(comparison))
pd.DataFrame(comparison).to_csv(output / "query_variants_codex.csv", index=False)

Single-query K100/M25 score 34.57228915662651 wrong 0


,band,photos,score60,wrong,answered,retrieval_share,kept_share,accepted_within1,median_inliers
0,100-300,74,47.351,0,71,1.000,1.000,44,49.5
1,300-500,48,47.875,0,48,0.958,0.958,29,81.5
2,500-800,68,50.471,0,67,1.000,1.000,44,61.0
3,800-1000,12,24.000,0,7,1.000,1.000,3,38.5
4,1000-1300,43,21.488,0,24,0.884,0.884,4,15.0
5,1300-1600,45,21.733,0,22,0.933,0.933,10,5.0
6,1600-2100,42,1.286,0,2,0.643,0.643,0,0.0


,variant,photos,score60,wrong,answered
0,plain (heading 0),332,34.572289,0,241
1,"3 headings (-5, 0, +5)",332,35.331325,0,245
2,5 headings (step 5),332,35.331325,0,245
3,9 headings (step 2.5),332,35.204819,0,247
4,"standard 6 (-5, 0, +5 x 1.0, 1.05)",332,35.493976,0,247
5,wide 27 (9 headings x 3 scales),332,35.963855,0,251


In [3]:
samples = select_visual_samples(development)
samples.to_csv(output / "viewed_development_images_codex.csv", index=False)
figure, axes = plt.subplots(7, 2, figsize=(10, 20))
for index, row in samples.iterrows():
    raw, view = read_view(images / row.filename)
    axis = axes.flat[index]
    axis.imshow(view, cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"{row.alt:.0f} m | {row.filename[:8]} | {row.band}")
    axis.axis("off")
figure.suptitle("Development drone images — independent per-image 1–99% stretch", fontsize=14)
figure.tight_layout(rect=(0, 0, 1, .98))
figure.savefig(output / "drone_images_by_height_codex.png", dpi=140)
plt.show()

<ipython-input-1-08cd8d2bec51>:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [4]:
focal = float(development.cam_fx.median())
physics = pd.DataFrame([sample_physics(h, focal) for h in [200, 500, 800, 1000, 1400, 1600, 2000, 2500, 3000]])
physics.to_csv(output / "resolution_by_height_codex.csv", index=False)
display(physics.round(2))
print("Nominal level11 cap starts limiting footprint selection above", 224*.3*2**(11/2)*focal/320, "metres; ignores tilt/yaw/height-scale corrections")
figure, axes = plt.subplots(1, 2, figsize=(12,4))
summary.plot(x="band", y=["retrieval_share", "kept_share"], kind="bar", ax=axes[0], ylim=(0,1), title="Single-query retrieval vs retained candidates")
summary.assign(answered_share=summary.answered/summary.photos).plot(x="band", y="answered_share", kind="bar", ax=axes[1], ylim=(0,1), title="Accepted share (single query)")
figure.tight_layout()
figure.savefig(output / "failure_stages_by_height_codex.png", dpi=150)
plt.show()

Nominal level11 cap starts limiting footprint selection above 1923.56332009375 metres; ignores tilt/yaw/height-scale corrections


<ipython-input-1-40f4f729b509>:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


,height_m,photo_gsd_m,footprint_width_m,requested_level,capped_level,canvas_gsd_m,coarse_grid_m,scoring_sigma_m,coarse_grid_sigma
0,200,0.99,316.20,5,5,1.70,6.79,8.53,0.80
1,500,2.47,790.49,8,8,4.80,19.20,10.91,1.76
2,800,3.95,1264.79,9,9,6.79,27.15,14.30,1.90
3,1000,4.94,1580.99,10,10,9.60,38.40,16.84,2.28
4,1400,6.92,2213.38,11,11,13.58,54.31,22.24,2.44
5,1600,7.90,2529.58,11,11,13.58,54.31,25.03,2.17
6,2000,9.88,3161.97,12,11,13.58,54.31,30.70,1.77
7,2500,12.35,3952.46,12,11,13.58,54.31,37.91,1.43
8,3000,14.82,4742.96,13,11,13.58,54.31,45.18,1.20


In [5]:
refinements = []
for name in ["highalt_dev332", "official500", "nudt300"]:
    table = pd.read_csv(source / f"coarse_refine/refine_{name}.csv")
    best = table.loc[table.groupby("photo").z1.idxmax()]
    groups = [("all", best)]
    if name == "highalt_dev332":
        groups.append(("height>=800", best[best.height>=800]))
    for group_name, group in groups:
        for variant in ["c", "p", "f2p", "f4p"]:
            if "err_"+variant in group:
                refinements.append(dict(data=name, group=group_name, variant=variant, **summarise_coarse(group, "err_"+variant)))
refinement_summary = pd.DataFrame(refinements)
display(refinement_summary.round(3))
refinement_summary.to_csv(output / "coarse_refinement_rescore_codex.csv", index=False)
confidence = []
for name in ["highalt_dev332", "official500", "nudt300"]:
    table = pd.read_csv(source / f"coarse_confidence/photos_{name}.csv")
    wrong = table[table.err>8]
    confidence.append(dict(data=name, photos=len(table), wrong=len(wrong), maximum_z_among_wrong=float(wrong.z.max())))
display(pd.DataFrame(confidence))
pd.DataFrame(confidence).to_csv(output / "coarse_confidence_transfer_codex.csv", index=False)
truth = pd.read_csv(source / "highalt_truth_diag/var_base.csv")
assert set(truth.filename).issubset(set(development.filename))
truth_bands = []
for band, group in add_height_band(truth, "alt").groupby("height_band", observed=True):
    truth_bands.append(dict(band=str(band), photos=len(group), truth_has_zero_inliers=int((group.inl_truth==0).sum()), truth_in_M=float(group.truth_in_M.mean()), median_truth_inliers=float(group.inl_truth.median())))
display(pd.DataFrame(truth_bands))
pd.DataFrame(truth_bands).to_csv(output / "truth_known_matcher_by_height_codex.csv", index=False)

,data,group,variant,photos,mean_points10,score60,wrong,within1_share
0,highalt_dev332,all,c,332,5.705,34.229,16,0.398
1,highalt_dev332,all,p,332,5.843,35.060,16,0.410
2,highalt_dev332,all,f2p,332,6.166,36.994,16,0.452
3,highalt_dev332,all,f4p,332,6.018,36.108,20,0.500
4,highalt_dev332,height>=800,c,142,3.127,18.761,16,0.246
5,highalt_dev332,height>=800,p,142,3.099,18.592,16,0.218
6,highalt_dev332,height>=800,f2p,142,3.634,21.803,16,0.289
7,highalt_dev332,height>=800,f4p,142,3.232,19.394,20,0.366
8,official500,all,c,500,6.264,37.584,52,0.742
9,official500,all,p,500,6.648,39.888,52,0.838


,data,photos,wrong,maximum_z_among_wrong
0,highalt_dev332,332,16,5.763279
1,official500,500,52,14.956482
2,nudt300,300,72,15.923258


,band,photos,truth_has_zero_inliers,truth_in_M,median_truth_inliers
0,800-1000,12,2,1.000000,26.0
1,1000-1300,43,14,0.790698,10.0
2,1300-1600,45,19,0.911111,5.0
3,1600-2100,42,38,0.642857,0.0


In [6]:
paths = [images / "metadata.csv", source / "highalt_split.csv", stage_path]
paths += list((source / "coarse_refine").glob("*.csv"))
paths += list((source / "coarse_confidence").glob("photos_*.csv"))
paths += [images / name for name in samples.filename]
manifest = []
for path in paths:
    manifest.append(dict(path=str(path.relative_to(root)), sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
(output / "inputs_codex.json").write_text(json.dumps(manifest, indent=2))
print("Saved review tables, figures and input hashes. No weights fitted or thresholds selected.")

Saved review tables, figures and input hashes. No weights fitted or thresholds selected.
